In [19]:
from google.adk.agents import SequentialAgent, LlmAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

from google.genai import types


In [12]:
# The first agent generates the initial draft.
generator = LlmAgent(
    name="DraftWriter",
    model="gemini-2.5-flash-lite",
    description="Generates initial draft content on a given subject.",
    instruction="Write a short, informative paragraph about the user's subject.",
    output_key="draft_text" # The output is saved to this state key.
)

# The second agent critiques the draft from the first agent.
reviewer = LlmAgent(
    name="FactChecker",
    model="gemini-2.5-flash-lite",
    description="Reviews a given text for factual accuracy and provides a structured critique.",
    instruction="""
    You are a meticulous fact-checker.
    1. Read the text provided in the state key 'draft_text'.
    2. Carefully verify the factual accuracy of all claims.
    3. Your final output must be a dictionary containing two keys:
       - "status": A string, either "ACCURATE" or "INACCURATE".
       - "reasoning": A string providing a clear explanation for your status, citing specific issues if any are found.
    """,
    output_key="review_output" # The structured dictionary is saved here.
)

In [13]:
# The SequentialAgent ensures the generator runs before the reviewer.
review_pipeline = SequentialAgent(
    name="WriteAndReview_Pipeline",
    sub_agents=[generator, reviewer]
)

# Execution Flow:
# 1. generator runs -> saves its paragraph to state['draft_text'].
# 2. reviewer runs -> reads state['draft_text'] and saves its dictionary output to state['review_output'].

In [17]:
# --- 3. ASYNC JUPYTER EXECUTION ---
print("--- Starting Sequential Pipeline ---")

session_service = InMemorySessionService()
await session_service.create_session(
    app_name="pipeline_app", 
    user_id="user_1", 
    session_id="session_1"
)

runner = Runner(
    agent=review_pipeline,
    app_name="pipeline_app",
    session_service=session_service
)

--- Starting Sequential Pipeline ---


In [20]:
# Trigger the sequence
msg = types.Content(role="user", parts=[types.Part(text="Start the generation process.")])

print("Running pipeline... (Waiting for both Generator and Reviewer to finish)")

async for event in runner.run_async(user_id="user_1", session_id="session_1", new_message=msg):
    if event.is_final_response():
        # The final response of the pipeline is usually the output of the LAST agent
        response_text = "".join([p.text for p in event.content.parts if p.text])
        print(f"\n🤖 REVIEWER FINAL OUTPUT:\n{response_text}")

# --- 4. INSPECT THE STATE (Proof it worked) ---
print("\n--- 🕵️‍♂️ Peeking into the Session State ---")
session = await session_service.get_session(app_name="pipeline_app", user_id="user_1", session_id="session_1")

print(f"📝 Generator's Draft:\n{session.state.get('draft_text', 'Not found')}\n")
print(f"🎯 Reviewer's Critique:\n{session.state.get('review_output', 'Not found')}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Running pipeline... (Waiting for both Generator and Reviewer to finish)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



🤖 REVIEWER FINAL OUTPUT:
The user has initiated the generation process. I am ready to receive a subject and will then generate a short, informative paragraph about it.

🤖 REVIEWER FINAL OUTPUT:
```json
{
 "status": "ACCURATE",
 "reasoning": "The provided text is an introductory statement indicating readiness to receive a subject for text generation. It contains no factual claims that can be verified or falsified."
}
```

--- 🕵️‍♂️ Peeking into the Session State ---
📝 Generator's Draft:
The user has initiated the generation process. I am ready to receive a subject and will then generate a short, informative paragraph about it.

🎯 Reviewer's Critique:
```json
{
 "status": "ACCURATE",
 "reasoning": "The provided text is an introductory statement indicating readiness to receive a subject for text generation. It contains no factual claims that can be verified or falsified."
}
```
